In [ ]:
from pydantic import BaseModel,Field, ConfigDict,PrivateAttr

from typing import Any
import pandas as pd
from abc import ABC, abstractmethod
from typing import Iterator,Callable

In [ ]:
from pathlib import Path
import pandas as pd

def listar_parquets(pasta: str):
    pasta_path = Path(pasta)
    arquivos = sorted(pasta_path.rglob("*.parquet"))
    return arquivos

def ler_parquets_em_um_df(pasta: str) -> pd.DataFrame:
    arquivos = listar_parquets(pasta)
    if not arquivos:
        return pd.DataFrame()

    # Lê todos e concatena em um único DataFrame
    dfs = [pd.read_parquet(arq) for arq in arquivos]
    df = pd.concat(dfs, ignore_index=True)

    return df

# Exemplo de uso:
pasta_base = r"/home/bruno.martins/dataset"  # ou r"C:\dados\minha_pasta"
arquivos_parquet = listar_parquets(pasta_base)
df = ler_parquets_em_um_df(pasta_base)

print(f"Encontrados {len(arquivos_parquet)} parquets.")
print(df.shape)


In [ ]:
from momentfm import MOMENTPipeline

model = MOMENTPipeline.from_pretrained(
    "AutonLab/MOMENT-1-large", 
    model_kwargs={
        'task_name': 'forecasting',
        'forecast_horizon': 48,
        'head_dropout': 0.1,
        'weight_decay': 0,
        'freeze_encoder': True, # Freeze the patch embedding layer
        'freeze_embedder': True, # Freeze the transformer encoder
        'freeze_head': False, # The linear forecasting head must be trained
    },
    # local_files_only=True,  # Whether or not to only look at local files (i.e., do not try to download the model).
)

In [ ]:
print("Unfrozen parameters:")
for name, param in model.named_parameters():    
    if param.requires_grad:
        print('    ', name)

In [ ]:
from pprint import pprint
import torch

# takes in tensor of shape [batchsize, n_channels, context_length]
x = torch.randn(16, 1, 512)
output = model(x_enc=x)
pprint(output)

In [ ]:
# ==== Tudo em UMA célula: leitura de parquets por percentual (com corte contínuo por timestamp)
#      + agrupamento por "objeto" (id pelo nome do arquivo ignorando _parteN/-parteN)
#      + segmentação por gaps
#      + InformerDataset que NÃO mistura objetos nem atravessa gaps ====

from typing import Optional, List, Dict, Tuple
import os
import re

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


# ---------- 1) Utilidades de arquivos / id de objeto ----------

def listar_parquets(pasta: str) -> List[str]:
    """Lista .parquet na pasta, ordenados por nome (determinístico)."""
    return sorted(
        os.path.join(pasta, f)
        for f in os.listdir(pasta)
        if f.lower().endswith(".parquet")
    )


def normalizar_id_objeto(caminho_arquivo: str) -> str:
    """
    Define o 'objeto' pelo nome-base do arquivo, removendo sufixos no final:
      _parte1, -parte1, _parte_2, -parte-03, etc.
    """
    nome = os.path.splitext(os.path.basename(caminho_arquivo))[0]
    nome = re.sub(r"([_-])parte([_-])?\d+$", "", nome, flags=re.IGNORECASE)
    return nome


# ---------- 2) Leitura por percentual com corte contínuo dentro de UM parquet ----------

def ler_parquets_em_um_df_percentual(
    pasta: str,
    percentual: float = 1.0,         # (0, 1]
    modo: str = "start",             # "start" (mais antigos) ou "end" (mais recentes) no arquivo parcial
    timestamp_col: str = "timestamp",
) -> pd.DataFrame:
    """
    Lê dados de uma pasta com parquets aplicando:
    - pega arquivos inteiros até atingir o alvo;
    - se precisar cortar, corta APENAS o último parquet necessário;
    - o corte é um trecho contínuo dentro do parquet com base no ordenamento por timestamp.

    Além disso adiciona:
    - __source_file : nome do parquet de origem
    - __object_id   : id normalizado do "objeto" (série) derivado do nome do arquivo
    """
    arquivos = listar_parquets(pasta)
    if not arquivos:
        return pd.DataFrame()

    if not (0 < percentual <= 1.0):
        raise ValueError("percentual deve estar em (0, 1].")

    # 1) Primeira passada: contar linhas (lê cada parquet uma vez)
    tamanhos = []
    total = 0
    for arq in arquivos:
        df_tmp = pd.read_parquet(arq)
        n = len(df_tmp)
        tamanhos.append(n)
        total += n

    alvo = max(1, int(total * percentual))

    # 2) Segunda passada: coletar até atingir o alvo
    dfs = []
    acumulado = 0

    for arq, n in zip(arquivos, tamanhos):
        if acumulado >= alvo:
            break

        df_parq = pd.read_parquet(arq)

        # garantir timestamp como coluna (se vier como índice)
        if timestamp_col not in df_parq.columns:
            df_parq = df_parq.reset_index()
            if "index" in df_parq.columns and timestamp_col not in df_parq.columns:
                df_parq = df_parq.rename(columns={"index": timestamp_col})

        # adicionar metadados
        df_parq["__source_file"] = os.path.basename(arq)
        df_parq["__object_id"] = normalizar_id_objeto(arq)

        falta = alvo - acumulado

        if len(df_parq) <= falta:
            dfs.append(df_parq)
            acumulado += len(df_parq)
        else:
            # parquet parcial: pegar trecho contínuo baseado em timestamp
            if timestamp_col not in df_parq.columns:
                raise ValueError(
                    f"Não encontrei a coluna {timestamp_col} no parquet {arq} "
                    "e não foi possível derivá-la do índice."
                )

            # ordena por timestamp para garantir 'trecho contínuo' coerente
            df_parq = df_parq.copy()
            df_parq[timestamp_col] = pd.to_datetime(df_parq[timestamp_col], errors="coerce")
            df_parq = df_parq.dropna(subset=[timestamp_col])
            df_parq = df_parq.sort_values(timestamp_col, kind="mergesort")

            # se após dropna ficou menor, ajusta falta para não estourar
            falta = min(falta, len(df_parq))

            if modo == "start":
                parte = df_parq.iloc[:falta]
            elif modo == "end":
                parte = df_parq.iloc[-falta:]
            else:
                raise ValueError("modo deve ser 'start' ou 'end'.")

            dfs.append(parte)
            acumulado += len(parte)
            break

    if not dfs:
        return pd.DataFrame()

    return pd.concat(dfs, ignore_index=True)


# ---------- 3) Segmentação por gaps (evita janelas atravessando saltos) ----------

def segmentar_por_gaps(
    df: pd.DataFrame,
    timestamp_col: str = "timestamp",
    gap_factor: float = 3.0,
) -> List[pd.DataFrame]:
    """
    Ordena por timestamp e divide em segmentos quando detecta gaps.
    freq = mediana(delta_t). Gap se delta_t > freq * gap_factor.
    """
    df = df.copy()
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce")
    df = df.dropna(subset=[timestamp_col])
    df = df.sort_values(timestamp_col, kind="mergesort").reset_index(drop=True)

    if len(df) < 2:
        return [df]

    deltas_ns = (
        df[timestamp_col]
        .diff()
        .dropna()
        .values.astype("timedelta64[ns]")
        .astype(np.int64)
    )

    freq = np.median(deltas_ns) if len(deltas_ns) else 0
    if freq <= 0:
        # timestamps iguais ou problemas de ordem; não segmenta
        return [df]

    is_gap = np.zeros(len(df), dtype=bool)
    is_gap[1:] = deltas_ns > (freq * gap_factor)

    starts = np.where(is_gap)[0]
    segments = []
    s0 = 0
    for s in starts:
        segments.append(df.iloc[s0:s].reset_index(drop=True))
        s0 = s
    segments.append(df.iloc[s0:].reset_index(drop=True))

    return [seg for seg in segments if len(seg) > 0]


# ---------- 4) InformerDataset que NÃO mistura objetos e NÃO cruza gaps ----------

class InformerDataset:
    def __init__(
        self,
        forecast_horizon: Optional[int] = 192,
        data_split: str = "train",           # "train" ou "test"
        data_stride_len: int = 1,
        task_name: str = "forecasting",      # "forecasting" ou "imputation"
        random_seed: int = 42,

        # leitura de parquets
        parquets_folder: Optional[str] = None,
        percentual: float = 1.0,
        modo_amostra: str = "start",
        timestamp_col: str = "timestamp",

        # tratamento
        drop_cols: Optional[List[str]] = None,     # ex.: ["date"]
        gap_factor: float = 3.0,
        train_ratio: float = 0.7,                  # split temporal dentro de cada segmento/objeto
    ):
        self.seq_len = 512
        self.forecast_horizon = forecast_horizon
        self.data_split = data_split
        self.data_stride_len = data_stride_len
        self.task_name = task_name
        self.random_seed = random_seed

        self.parquets_folder = parquets_folder
        self.percentual = percentual
        self.modo_amostra = modo_amostra
        self.timestamp_col = timestamp_col
        self.drop_cols = drop_cols or []
        self.gap_factor = gap_factor
        self.train_ratio = train_ratio

        self._read_data()

    def _read_data(self):
        if not self.parquets_folder:
            raise ValueError("parquets_folder deve ser informado.")

        # 1) lê os parquets com lógica do percentual
        df_all = ler_parquets_em_um_df_percentual(
            pasta=self.parquets_folder,
            percentual=self.percentual,
            modo=self.modo_amostra,
            timestamp_col=self.timestamp_col,
        )

        if df_all.empty:
            raise ValueError("Nenhum dado foi carregado (pasta vazia ou percentual muito pequeno).")

        # 2) drop colunas indesejadas (se existirem)
        for c in self.drop_cols:
            if c in df_all.columns:
                df_all = df_all.drop(columns=[c])

        # 3) valida colunas internas
        if "__object_id" not in df_all.columns:
            raise ValueError("Coluna interna __object_id não foi criada.")
        if self.timestamp_col not in df_all.columns:
            raise ValueError(f"Coluna {self.timestamp_col} não encontrada após leitura.")

        self.length_timeseries_original = len(df_all)

        # 4) definir feature_cols (todas exceto timestamp e colunas internas)
        internal_cols = {self.timestamp_col, "__source_file", "__object_id"}
        feature_cols = [c for c in df_all.columns if c not in internal_cols]
        if not feature_cols:
            raise ValueError("Não encontrei colunas de features (numéricas) para o modelo.")

        self.feature_cols = feature_cols
        self.n_channels = len(self.feature_cols)

        # 5) construir segmentos por objeto (evita mistura de objetos)
        self.segments: Dict[str, List[pd.DataFrame]] = {}
        for obj_id, df_obj in df_all.groupby("__object_id", sort=False):
            segs = segmentar_por_gaps(
                df_obj,
                timestamp_col=self.timestamp_col,
                gap_factor=self.gap_factor,
            )
            self.segments[obj_id] = segs

        # 6) ajustar scaler SOMENTE usando a parte de treino de cada segmento (temporal)
        self.scaler = StandardScaler()
        train_values = []

        for obj_id, segs in self.segments.items():
            for seg in segs:
                seg = seg.copy()
                seg[self.feature_cols] = seg[self.feature_cols].infer_objects(copy=False)
                seg[self.feature_cols] = seg[self.feature_cols].interpolate(method="cubic")

                T = len(seg)
                cut = int(T * self.train_ratio)
                if cut <= 0:
                    continue

                train_values.append(seg.iloc[:cut][self.feature_cols].values)

        if not train_values:
            raise ValueError("Não há dados suficientes para ajustar o scaler (treino vazio).")

        self.scaler.fit(np.vstack(train_values))

        # 7) transformar segmentos e criar índice de janelas válidas por split
        self.data_by_segment: Dict[Tuple[str, int], np.ndarray] = {}
        self.windows: List[Tuple[str, int, int]] = []  # (obj_id, seg_id, start)

        if self.task_name == "forecasting":
            need = self.seq_len + self.forecast_horizon
        elif self.task_name == "imputation":
            need = self.seq_len
        else:
            raise ValueError("task_name deve ser 'forecasting' ou 'imputation'.")

        for obj_id, segs in self.segments.items():
            for seg_id, seg in enumerate(segs):
                seg = seg.copy()
                seg[self.feature_cols] = seg[self.feature_cols].infer_objects(copy=False)
                seg[self.feature_cols] = seg[self.feature_cols].interpolate(method="cubic")

                arr = self.scaler.transform(seg[self.feature_cols].values)  # (T, C)
                self.data_by_segment[(obj_id, seg_id)] = arr

                T = arr.shape[0]
                cut = int(T * self.train_ratio)

                if self.data_split == "train":
                    start0 = 0
                    usable_end = cut
                elif self.data_split == "test":
                    start0 = cut
                    usable_end = T
                else:
                    raise ValueError("data_split deve ser 'train' ou 'test'.")

                last_start = usable_end - need
                if last_start < start0:
                    continue

                for s in range(start0, last_start + 1, self.data_stride_len):
                    self.windows.append((obj_id, seg_id, s))

        if not self.windows:
            raise ValueError(
                "Nenhuma janela válida foi gerada. "
                "Verifique percentual, seq_len, forecast_horizon, train_ratio e gap_factor."
            )

        self.length_timeseries = len(self.windows)  # aqui é o número de amostras/janelas

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, index):
        obj_id, seg_id, s = self.windows[index]
        arr = self.data_by_segment[(obj_id, seg_id)]

        seq_start = s
        seq_end = seq_start + self.seq_len
        input_mask = np.ones(self.seq_len)

        if self.task_name == "forecasting":
            pred_end = seq_end + self.forecast_horizon
            timeseries = arr[seq_start:seq_end, :].T
            forecast = arr[seq_end:pred_end, :].T
            return timeseries, forecast, input_mask

        # imputation
        timeseries = arr[seq_start:seq_end, :].T
        return timeseries, input_mask


# ---------- Exemplo de uso ----------
# ds_train = InformerDataset(
#     data_split="train",
#     task_name="forecasting",
#     parquets_folder=r"C:\caminho\para\pasta",
#     percentual=0.15,
#     modo_amostra="start",
#     timestamp_col="timestamp",
#     drop_cols=["date"],
#     gap_factor=3.0,
#     train_ratio=0.7,
# )
#
# x, y, mask = ds_train[0]
# print(x.shape, y.shape, mask.shape)


In [ ]:
import numpy as np
import torch
import torch.cuda.amp
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import OneCycleLR
from tqdm import tqdm

from momentfm.utils.utils import control_randomness
from momentfm.utils.forecasting_metrics import get_forecasting_metrics

# Set random seeds for PyTorch, Numpy etc.
control_randomness(seed=13) 

# Load data
train_dataset = InformerDataset(data_split="train", random_seed=13, forecast_horizon=48,percentual=0.25,parquets_folder=r"/home/bruno.martins/dataset")
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

test_dataset = InformerDataset(data_split="test", random_seed=13, forecast_horizon=48,percentual=0.25,parquets_folder=r"/home/bruno.martins/dataset")
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=True)

criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cur_epoch = 0
max_epoch = 1

# Move the model to the GPU
model = model.to(device)

# Move the loss function to the GPU
criterion = criterion.to(device)

# Enable mixed precision training
scaler = torch.cuda.amp.GradScaler()

# Create a OneCycleLR scheduler
max_lr = 1e-4
total_steps = len(train_loader) * max_epoch
scheduler = OneCycleLR(optimizer, max_lr=max_lr, total_steps=total_steps, pct_start=0.3)

# Gradient clipping value
max_norm = 5.0

while cur_epoch < max_epoch:
    losses = []
    for timeseries, forecast, input_mask in tqdm(train_loader, total=len(train_loader)):
        # Move the data to the GPU
        timeseries = timeseries.float().to(device)
        input_mask = input_mask.to(device)
        forecast = forecast.float().to(device)

        with torch.cuda.amp.autocast():
            output = model(x_enc=timeseries, input_mask=input_mask)
        
        loss = criterion(output.forecast, forecast)

        # Scales the loss for mixed precision training
        scaler.scale(loss).backward()

        # Clip gradients
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

        losses.append(loss.item())

    losses = np.array(losses)
    average_loss = np.average(losses)
    print(f"Epoch {cur_epoch}: Train loss: {average_loss:.3f}")

    # Step the learning rate scheduler
    scheduler.step()
    cur_epoch += 1
    
    # Evaluate the model on the test split
    trues, preds, histories, losses = [], [], [], []
    model.eval()
    with torch.no_grad():
        for timeseries, forecast, input_mask in tqdm(test_loader, total=len(test_loader)):
        # Move the data to the GPU
            timeseries = timeseries.float().to(device)
            input_mask = input_mask.to(device)
            forecast = forecast.float().to(device)

            with torch.cuda.amp.autocast():
                output = model(x_enc=timeseries, input_mask=input_mask)
            
            loss = criterion(output.forecast, forecast)                
            losses.append(loss.item())

            trues.append(forecast.detach().cpu().numpy())
            preds.append(output.forecast.detach().cpu().numpy())
            histories.append(timeseries.detach().cpu().numpy())
    
    losses = np.array(losses)
    average_loss = np.average(losses)
    model.train()

    trues = np.concatenate(trues, axis=0)
    preds = np.concatenate(preds, axis=0)
    histories = np.concatenate(histories, axis=0)
    
    metrics = get_forecasting_metrics(y=trues, y_hat=preds, reduction='mean')

    print(f"Epoch {cur_epoch}: Test MSE: {metrics.mse:.3f} | Test MAE: {metrics.mae:.3f}")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

# Pegue 1 batch do teste
timeseries, forecast, input_mask = next(iter(test_loader))

timeseries = timeseries.float().to(device)      # (B, C, T_hist) no seu caso: (8, 1, 512)
forecast   = forecast.float().to(device)        # (B, C, H)      no seu caso: (8, 1, 48)
input_mask = input_mask.to(device)              # (B, T_hist)    no seu caso: (8, 512)

with torch.no_grad():
    output = model(x_enc=timeseries, input_mask=input_mask)
    pred = output.forecast                       # (B, C, H)

# Escolha qual item do batch plotar
i = 0          # amostra dentro do batch
c = 0          # canal/variável

hist = timeseries[i, c].detach().cpu().numpy()   # (T_hist,)
true = forecast[i, c].detach().cpu().numpy()     # (H,)
yhat = pred[i, c].detach().cpu().numpy()         # (H,)

# Eixo de tempo "colado": histórico seguido de horizonte
t_hist = np.arange(len(hist))
t_fut  = np.arange(len(hist), len(hist) + len(true))

plt.figure(figsize=(12, 4))
plt.plot(t_hist, hist, label="Histórico (entrada)")
plt.plot(t_fut, true, label="Verdade (forecast)", linewidth=2)
plt.plot(t_fut, yhat,  label="Previsão (modelo)", linewidth=2)

plt.axvline(len(hist)-1, color="k", linestyle="--", linewidth=1)
plt.title("Teste: histórico vs verdade vs previsão")
plt.xlabel("índice de tempo")
plt.ylabel("valor")
plt.legend()
plt.tight_layout()
plt.show()
